# 红利低波原始指数编制

本 notebook 可从项目内任意目录运行。请按单元格分段执行：启动一个数据更新任务，按需控制或等待保存完成，再启动下一个任务。

In [1]:
import os
import sys
from pathlib import Path

search_paths = (Path.cwd(), *Path.cwd().parents)
project_root = next(path for path in search_paths if (path / "pyproject.toml").exists())
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd

from pyquant import load_dataset, update_dataset
from pyquant.io import load_config
from strategies.cross_sectional.dividend_low_vol.components import (
    calculate_dividend_low_vol_index,
    select_dividend_low_vol_constituents,
)

## 调仓日期

`rebalance_date` 同时作为选样信息截止日和新指数段起点。本策略不区分选样日与生效日。

In [3]:
rebalance_date = pd.Timestamp("2013-12-17")
end_date = pd.Timestamp("2023-06-16")
history_start = rebalance_date - pd.DateOffset(years=1)
download_pool: str | list[str] = "all"

## 数据更新

三个数据集共用下载锁，因此同一时间只能运行一个任务。每个启动单元格会立即返回；开始下一个任务前，先等待当前任务完成，或执行停止并保存。`download_pool` 可使用股票池名称，也可改为 BaoStock 证券代码列表。

In [ ]:
download_job = update_dataset(
    "stock_daily",
    start=history_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=download_pool,
)

Updated 5537/5537

login success!
logout success!


### 下载控制

进度由启动单元格自动显示。暂停和继续不会丢失数据；“停止并保存”会等待待写数据、下载锁和登录会话处理完毕。

#### 暂停

In [ ]:
download_job.pause()
download_job.state

#### 继续

In [ ]:
download_job.resume()
download_job.state

#### 停止并保存

In [6]:
download_job.stop()
download_job.wait()
download_job.state

'completed'

### 分红数据

确认上一个任务已经完成或停止保存后，再执行本单元格。

In [5]:
if download_job.state in {"running", "paused", "stopping"}:
    raise RuntimeError("Finish the active dataset update first")
download_job.wait()

download_job = update_dataset(
    "dividend",
    start=history_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=download_pool,
)

Updated 5537/5537

login success!
logout success!


### 季度总股本

确认上一个任务已经完成或停止保存后，再执行本单元格。

In [6]:
if download_job.state in {"running", "paused", "stopping"}:
    raise RuntimeError("Finish the active dataset update first")
download_job.wait()

download_job = update_dataset(
    "stock_profit_quarterly",
    start=history_start.strftime("%Y-%m-%d"),
    end=rebalance_date.strftime("%Y-%m-%d"),
    pool=download_pool,
)

Updated 5537/5537

login success!
logout success!


## 通用数据集读取

行情明确使用不复权口径；分红、查询覆盖和股本数据均通过数据集目录读取。

In [7]:
if download_job.state in {"running", "paused", "stopping"}:
    raise RuntimeError("Finish the active dataset update first")
download_job.wait()

strategy_dir = project_root / "strategies/cross_sectional/dividend_low_vol"
config = load_config(strategy_dir / "config.yaml")
price = load_dataset(
    "stock_daily",
    start=history_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
)
dividends = load_dataset("dividend")
dividend_queries = load_dataset("dividend_queries")
shares = load_dataset("stock_profit_quarterly")

ValueError: Dataset 'stock_daily' contains duplicate primary keys: ['date', 'symbol']

## 单次选样快照

In [ ]:
constituents = select_dividend_low_vol_constituents(
    price,
    dividends,
    dividend_queries,
    shares,
    rebalance_date,
    config,
)
constituents[
    [
        "price_date",
        "avg_dividend_yield_3y",
        "dividend_yield_rank",
        "volatility_240d",
        "volatility_rank",
        "weight",
    ]
].sort_values("weight", ascending=False)

## 单个调仓区间指数

首段两类指数均从 1000 点起算。未来拼接调仓区间时，将上一段调仓日收盘点位分别作为下一段的两个基点，并删除重复边界行。

In [ ]:
index_segment = calculate_dividend_low_vol_index(
    price,
    dividends,
    dividend_queries,
    constituents,
    rebalance_date,
    end_date,
)
index_segment.tail()

In [ ]:
index_segment[["price_index", "total_return_index"]].plot(
    figsize=(12, 5),
    title="红利低波价格指数与全收益指数",
)